# Labwork 4 - Architecture: SOLID, DRY, KISS *(cleaned starter)*

Each exercise gives code that **violates one principle**. Refactor the file in place so the principle holds,
then check with `!python` and `!mypy --strict`. Solutions are in the `exercise*.py` correction files.

Exercises 3, 4 and 5 have you write interfaces. Follow the two conventions from Lecture 4: name each one with a leading capital `I` (`ICoffeeShop`, `IDeliverable`, ...), and keep it **pure** -- every method decorated `@abstractmethod`, with `...` for a body, no `__init__` and no attributes. If a class of yours needs to hold state or share code, it is an abstract base class, not an interface: leave it its plain name.

Run once to create the working directory.

In [1]:
import os, sys

# The interpreter that is running this notebook.
# On macOS there is often no `python` command at all, only `python3`, and adding a
# shell alias does not help: `!` cells run a NON-interactive shell that never reads
# your profile, and `%%script` launches the program directly, without any shell.
# `sys.executable` is an absolute path, so it keeps working after a `cd` too.
PY = sys.executable
os.makedirs('code', exist_ok=True)
print('code/ ready')
print(f'interpreter: {PY}')


code/ ready
interpreter: /usr/local/bin/python3


## Exercise 1 - SRP (Single Responsibility)
The class below mixes two responsibilities (identity and address). Split it so each class has one reason to change.

In [2]:
%%writefile code/exercise1.py
class Address:
    def __init__(self, city: str, zip_code: int) -> None:
        self.__city = city
        self.__zip_code = zip_code

    @property
    def city(self) -> str:
        return self.__city

    @property
    def zip_code(self) -> int:
        return self.__zip_code

    def change_address(self, city: str, zip_code: int) -> None:
        self.__city = city
        self.__zip_code = zip_code


class CoffeeShop:
    def __init__(self, name: str, address: Address) -> None:
        self.__name = name
        self.address = address

    @property
    def name(self) -> str:
        return self.__name


if __name__ == '__main__':
    # Initialize the separated Address and CoffeeShop objects
    addr = Address('Ha Noi', 111000)
    cs = CoffeeShop('La Viet', addr)
    
    # Update the address using the composed object
    cs.address.change_address('Neuville de Poitou', 86170)
    
    print(f'CoffeeShop name is "{cs.name}"')


Overwriting code/exercise1.py


In [3]:
!{PY} code/exercise1.py
!{PY} -m mypy --strict code/exercise1.py

CoffeeShop name is "La Viet"
Success: no issues found in 1 source file


## Exercise 2 - OCP (Open/Closed)
`InvoiceService` must be edited every time a new company is added. Refactor with polymorphism so a new company
needs **no change** to existing code.

In [6]:
%%writefile code/exercise2.py
from abc import ABC, abstractmethod

class Company(ABC):
    def __init__(self, name: str) -> None:
        self.__name = name

    @property
    def name(self) -> str:
        return self.__name
        
    @abstractmethod
    def generate_invoice(self) -> str:
        pass

class CompanyA(Company):
    def generate_invoice(self) -> str:
        return "some format of invoice for A company"

class CompanyB(Company):
    def generate_invoice(self) -> str:
        return "some format of invoice for B company"

class CompanyC(Company):
    def generate_invoice(self) -> str:
        return "some format of invoice for C company"

class CompanyD(Company):
    def generate_invoice(self) -> str:
        return "some format of invoice for D company"

class InvoiceService:
    @staticmethod
    def generate_invoice(company: Company) -> str:
        return company.generate_invoice()

if __name__ == "__main__":
    for company in (CompanyA('A'), CompanyB('B'), CompanyC('C'), CompanyD('D')):
        print(InvoiceService.generate_invoice(company))


Overwriting code/exercise2.py


In [7]:
!{PY} code/exercise2.py
!{PY} -m mypy --strict code/exercise2.py

some format of invoice for A company
some format of invoice for B company
some format of invoice for C company
some format of invoice for D company
Success: no issues found in 1 source file


## Exercise 3 - LSP (Liskov Substitution)
`B` breaks the contract of `CoffeeShop.takeaway` (it raises instead of honouring it). Refactor so every
`CoffeeShop` is substitutable - e.g. move the delivery capability into its own interface.

In [14]:
%%writefile code/exercise3.py
from abc import ABC, abstractmethod

class ITakeaway(ABC):
    @abstractmethod
    def takeaway(self) -> str:
        ...

class CoffeeShop:
    def __init__(self, name: str) -> None:
        self.__name = name

    @property
    def name(self) -> str:
        return self.__name

# A inherits from both CoffeeShop and the ITakeaway interface
class A(CoffeeShop, ITakeaway):
    def takeaway(self) -> str:
        return "Delivery at most 30 minutes"

# B is now perfectly substitutable for CoffeeShop without needing a fake takeaway method
class B(CoffeeShop):
    pass


if __name__ == "__main__":
    # using type narrowing (isinstance) to safely check for the capability
    # without breaking MyPy strict type hinting, or  just pass specific classes.
    def display(shop: CoffeeShop) -> None:
        if isinstance(shop, ITakeaway):
            print(f'CoffeeShop {shop.name}: {shop.takeaway()}')
        else:
            print(f'CoffeeShop {shop.name}: No takeaway service')

    display(A('A'))
    display(B('B'))


Overwriting code/exercise3.py


In [15]:
!{PY} code/exercise3.py
!{PY} -m mypy --strict code/exercise3.py

CoffeeShop A: Delivery at most 30 minutes
CoffeeShop B: No takeaway service
Success: no issues found in 1 source file


## Exercise 4 - ISP (Interface Segregation)
`ICoffeeShop` is too fat: classes are forced to `raise NotImplementedError`. Split it into segregated interfaces.

In [10]:
%%writefile code/exercise4.py
from abc import ABC, abstractmethod

class ITraditionalShop(ABC):
    @abstractmethod
    def brew_by_espresso_machine(self) -> None: ...
    
    @abstractmethod
    def brew_machine_pour_over(self) -> None: ...


class IThirdWaveShop(ABC):
    @abstractmethod
    def brew_by_hand_held_espresso_maker(self) -> None: ...
    
    @abstractmethod
    def brew_manual_pour_over(self) -> None: ...


class IFilterCoffee(ABC):
    @abstractmethod
    def brew_filter_coffee(self) -> None: ...


# Traditional now only inherits the interfaces it actually implements
class Traditional(ITraditionalShop, IFilterCoffee):
    def brew_by_espresso_machine(self) -> None:
        print("brewing by espresso machine")

    def brew_machine_pour_over(self) -> None:
        print("brewing machine pour over")

    def brew_filter_coffee(self) -> None:
        print("brewing filter coffee")


if __name__ == "__main__":
    Traditional().brew_filter_coffee()


Writing code/exercise4.py


In [11]:
!{PY} code/exercise4.py
!{PY} -m mypy --strict code/exercise4.py

brewing filter coffee
Success: no issues found in 1 source file


## Exercise 5 - DIP (Dependency Inversion)
`Delivery` depends on the concrete `Customer` and `CoffeeShop`. Refactor so it depends on abstractions (interfaces).

In [12]:
%%writefile code/exercise5.py
from abc import ABC, abstractmethod

#Define Abstractions (Interfaces)
class ICustomer(ABC):
    @abstractmethod
    def make_payment(self) -> None: ...
    
    @abstractmethod
    def receive_coffee(self) -> None: ...


class ICoffeeShop(ABC):
    @abstractmethod
    def get_payment(self) -> None: ...
    
    @abstractmethod
    def deliver_coffee(self) -> None: ...


#Implement Abstractions in Concrete Classes
class CoffeeShop(ICoffeeShop):
    def __init__(self, name: str) -> None:
        self.__name = name

    def get_payment(self) -> None:
        print(f"{self.__name} gets the payment")

    def deliver_coffee(self) -> None:
        print(f"{self.__name} delivers the coffee")


class Customer(ICustomer):
    def __init__(self, name: str) -> None:
        self.__name = name

    def make_payment(self) -> None:
        print(f"{self.__name} makes the payment")

    def receive_coffee(self) -> None:
        print(f"{self.__name} receives the coffee")


#Depend on Abstractions
class Delivery:
    def __init__(self, customer: ICustomer, coffee_shop: ICoffeeShop) -> None:
        self.__customer = customer
        self.__coffee_shop = coffee_shop

    def delivers(self) -> None:
        self.__customer.make_payment()
        self.__coffee_shop.get_payment()
        self.__coffee_shop.deliver_coffee()
        self.__customer.receive_coffee()


if __name__ == '__main__':
    Delivery(Customer('Uncle Bob'), CoffeeShop('La Viet')).delivers()


Writing code/exercise5.py


In [13]:
!{PY} code/exercise5.py
!{PY} -m mypy --strict code/exercise5.py

Uncle Bob makes the payment
La Viet gets the payment
La Viet delivers the coffee
Uncle Bob receives the coffee
Success: no issues found in 1 source file


*Reminder: KISS (keep it simple) and DRY (don't repeat yourself) apply throughout; and **YAGNI** - you aren't gonna need it - keep interfaces only as wide as today's need.*